<a href="https://colab.research.google.com/github/Yiliyiliyiliyiliyili/Testing-the-Theoretical-Claims-of-LeJEPA-by-JAX/blob/main/experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q equinox optax

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import numpy as np
import pickle
from itertools import combinations
import matplotlib.pyplot as plt
import pandas as pd

print("JAX version:", jax.__version__)
print("Devices    :", jax.devices())

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

tf.config.set_visible_devices([], "GPU")


def _augment(img):
    """
    SimCLR-style augmentation for a single (32, 32, 3) uint8 image.
    Returns (3, 32, 32) float32 CHW tensor.
    """
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.image.pad_to_bounding_box(img, 4, 4, 40, 40)
    img = tf.image.random_crop(img, [32, 32, 3])
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.4)
    img = tf.image.random_contrast(img, lower=0.6, upper=1.4)
    img = tf.image.random_saturation(img, lower=0.6, upper=1.4)
    img = tf.image.random_hue(img, max_delta=0.1)
    img = tf.clip_by_value(img, 0.0, 1.0)
    img = tf.cond(
        tf.random.uniform(()) < 0.2,
        lambda: tf.repeat(tf.image.rgb_to_grayscale(img), 3, axis=-1),
        lambda: img,
    )
    mean = tf.constant([0.4914, 0.4822, 0.4465])
    std  = tf.constant([0.2023, 0.1994, 0.2010])
    img  = (img - mean) / std
    return tf.transpose(img, [2, 0, 1])


def make_train_loader(batch_size, n_views=4):
    """
    Infinite iterator of (N, V, C, H, W) JAX arrays for SSL training.
    Uses tf.data prefetch so CPU prepares next batch while GPU runs current step.
    """
    ds = tfds.load("cifar10", split="train", as_supervised=True)
    ds = ds.map(
        lambda img, _: tf.stack([_augment(img) for _ in range(n_views)]),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = (ds
          .shuffle(10_000)
          .batch(batch_size, drop_remainder=True)
          .prefetch(tf.data.AUTOTUNE)
          .repeat())
    return map(jnp.array, tfds.as_numpy(ds))


def make_eval_loader(split, batch_size=512):
    """
    Finite iterator of (imgs, labels) for linear probe evaluation.
    No augmentation — normalize only.
    """
    def preprocess(img, label):
        img  = tf.cast(img, tf.float32) / 255.0
        mean = tf.constant([0.4914, 0.4822, 0.4465])
        std  = tf.constant([0.2023, 0.1994, 0.2010])
        img  = (img - mean) / std
        return tf.transpose(img, [2, 0, 1]), label

    ds = tfds.load("cifar10", split=split, as_supervised=True)
    ds = (ds
          .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(batch_size, drop_remainder=False)
          .prefetch(tf.data.AUTOTUNE))
    return tfds.as_numpy(ds)


# Shape checks
_loader = make_train_loader(batch_size=4)
_batch  = next(_loader)
print("train batch shape:", _batch.shape)
assert _batch.shape == (4, 4, 3, 32, 32)
_imgs, _labels = next(iter(make_eval_loader("test", batch_size=8)))
print("eval batch shape :", _imgs.shape)
assert _imgs.shape == (8, 3, 32, 32)
del _loader, _batch, _imgs, _labels
print("Data pipeline check passed.")


In [ ]:
def instance_norm(x, eps: float = 1e-5):
    """
    Stateless InstanceNorm for a single (C, H, W) feature map.
    Normalizes over spatial dims independently per channel.
    Chosen over BatchNorm to keep the model a pure function (no mutable state),
    which is required for JAX/Equinox compatibility.
    """
    mean = jnp.mean(x, axis=(1, 2), keepdims=True)
    var  = jnp.var( x, axis=(1, 2), keepdims=True)
    return (x - mean) / jnp.sqrt(var + eps)


class ResBlock(eqx.Module):
    """
    Standard residual block with InstanceNorm.
    shortcut is always created regardless of use_shortcut to keep the pytree
    structure fixed, avoiding JIT recompilation across different block configs.
    """
    conv1:        eqx.nn.Conv2d
    conv2:        eqx.nn.Conv2d
    shortcut:     eqx.nn.Conv2d
    use_shortcut: bool = eqx.field(static=True)

    def __init__(self, in_ch: int, out_ch: int, stride: int, key):
        k1, k2, k3 = jax.random.split(key, 3)
        self.conv1    = eqx.nn.Conv2d(in_ch,  out_ch, 3, stride=stride,
                                       padding=1, use_bias=False, key=k1)
        self.conv2    = eqx.nn.Conv2d(out_ch, out_ch, 3, stride=1,
                                       padding=1, use_bias=False, key=k2)
        self.shortcut = eqx.nn.Conv2d(in_ch,  out_ch, 1, stride=stride,
                                       use_bias=False, key=k3)
        self.use_shortcut = (stride != 1 or in_ch != out_ch)

    def __call__(self, x):
        out = jax.nn.relu(instance_norm(self.conv1(x)))
        out = instance_norm(self.conv2(out))
        res = self.shortcut(x) if self.use_shortcut else x
        return jax.nn.relu(out + res)


class MiniResNet18(eqx.Module):
    """
    Lightweight ResNet-18 for CIFAR-10 (32x32 input).

    Stem:   Conv(3->64, 3x3) + InstanceNorm + ReLU
    Stage1: ResBlock(64->64,   stride=1) x2
    Stage2: ResBlock(64->128,  stride=2) x2   [32->16]
    Stage3: ResBlock(128->256, stride=2) x2   [16->8]
    Stage4: ResBlock(256->512, stride=2) x2   [8->4]
    GlobalAvgPool -> ProjMLP: 512->512->512->64

    encode(): returns 512-dim backbone features, used for linear probe.
    __call__(): returns 64-dim projected embeddings, used during SSL training.
    """
    stem:   eqx.nn.Conv2d
    blocks: list
    fc1:    eqx.nn.Linear
    fc2:    eqx.nn.Linear
    fc3:    eqx.nn.Linear

    def __init__(self, key):
        ks = jax.random.split(key, 12)
        self.stem = eqx.nn.Conv2d(3, 64, 3, padding=1, use_bias=False, key=ks[0])
        self.blocks = [
            ResBlock(64,  64,  1, ks[1]),
            ResBlock(64,  64,  1, ks[2]),
            ResBlock(64,  128, 2, ks[3]),
            ResBlock(128, 128, 1, ks[4]),
            ResBlock(128, 256, 2, ks[5]),
            ResBlock(256, 256, 1, ks[6]),
            ResBlock(256, 512, 2, ks[7]),
            ResBlock(512, 512, 1, ks[8]),
        ]
        self.fc1 = eqx.nn.Linear(512, 512, key=ks[9])
        self.fc2 = eqx.nn.Linear(512, 512, key=ks[10])
        self.fc3 = eqx.nn.Linear(512, 64,  key=ks[11])

    def encode(self, x):
        """Backbone only. x: (C, H, W) -> (512,). Used for linear probe."""
        x = jax.nn.relu(instance_norm(self.stem(x)))
        for block in self.blocks:
            x = block(x)
        return jnp.mean(x, axis=(1, 2))

    def __call__(self, x):
        """Full forward pass. x: (C, H, W) -> (64,). Used during SSL training."""
        x = self.encode(x)
        x = jax.nn.relu(self.fc1(x))
        x = jax.nn.relu(self.fc2(x))
        return self.fc3(x)


def forward(model, x):
    """
    Encode a multi-view batch.
    x: (N, V, C, H, W) -> z: (V, N, D)
    Flattens views into batch dim, vmaps encoder, then restores shape.
    """
    N, V, C, H, W = x.shape
    z_flat = jax.vmap(model)(x.reshape(N * V, C, H, W))
    return jnp.swapaxes(z_flat.reshape(N, V, -1), 0, 1)


# Shape checks
_m = MiniResNet18(jax.random.PRNGKey(99))
_z = forward(_m, jnp.zeros((4, 4, 3, 32, 32)))
assert _z.shape == (4, 4, 64), f"forward shape mismatch: {_z.shape}"
_f = jax.vmap(_m.encode)(jnp.zeros((8, 3, 32, 32)))
assert _f.shape == (8, 512), f"encode shape mismatch: {_f.shape}"
del _m, _z, _f
print("Model architecture check passed.")


In [ ]:
def sigreg_loss(z, key, n_dirs: int = 512, n_t: int = 10):
    """
    Epps-Pulley characteristic function distance between the projected
    embedding distribution and N(0,1).

    For each of n_dirs random unit directions d:
        proj = z_flat @ d
        dist = mean over t of:
               (E[cos(t*proj)] - exp(-t^2/2))^2 + (E[sin(t*proj)])^2

    n_dirs=512: paper shows 512 directions achieve results close to 1024
    at half the computational cost.
    Directions resampled each call for unbiased estimation and to prevent
    the optimizer from exploiting fixed directions.

    z: (V, N, D) -> scalar
    """
    V, N, D  = z.shape
    z_flat   = z.reshape(V * N, D)
    raw      = jax.random.normal(key, (n_dirs, D))
    dirs     = raw / jnp.linalg.norm(raw, axis=-1, keepdims=True)
    proj     = z_flat @ dirs.T
    t_grid   = jnp.linspace(-4.0, 4.0, n_t)
    phi_norm = jnp.exp(-0.5 * t_grid ** 2)
    proj_t   = proj[:, :, None] * t_grid[None, None, :]
    mean_cos = jnp.mean(jnp.cos(proj_t), axis=0)
    mean_sin = jnp.mean(jnp.sin(proj_t), axis=0)
    return jnp.mean((mean_cos - phi_norm[None, :]) ** 2 + mean_sin ** 2)


def vicreg_loss(z, alpha: float = 25.0, beta: float = 25.0, gamma: float = 1.0):
    """
    Original VICReg loss (Bardes et al. 2022) extended to V=4 views.
    Uses paper-recommended weights alpha=25, beta=25, gamma=1.

    Variance  : hinge loss penalizing std < 1 per dimension.
    Invariance: mean MSE over all C(V,2) view pairs.
    Covariance: off-diagonal covariance penalty.

    z: (V, N, D) -> scalar
    """
    V, N, D = z.shape
    pairs   = list(combinations(range(V), 2))
    inv     = sum(jnp.mean((z[i] - z[j]) ** 2) for i, j in pairs) / len(pairs)

    var_loss = jnp.array(0.0)
    cov_loss = jnp.array(0.0)
    for v in range(V):
        zv       = z[v]
        std      = jnp.sqrt(jnp.var(zv, axis=0) + 1e-4)
        var_loss = var_loss + jnp.mean(jax.nn.relu(1.0 - std))
        zv_c     = zv - jnp.mean(zv, axis=0, keepdims=True)
        cov      = (zv_c.T @ zv_c) / (N - 1)
        cov_loss = cov_loss + (
            jnp.sum(cov ** 2) - jnp.sum(jnp.diag(cov) ** 2)
        ) / D

    return alpha * (var_loss / V) + beta * inv + gamma * (cov_loss / V)


def invariance_loss(z):
    """Mean MSE over all C(V,2) view pairs. z: (V, N, D) -> scalar."""
    pairs = list(combinations(range(z.shape[0]), 2))
    return sum(jnp.mean((z[i] - z[j]) ** 2) for i, j in pairs) / len(pairs)


def lejepa_loss(z, key, lam: float):
    """
    LeJEPA total loss: lam * SIGReg(z) + (1 - lam) * Invariance(z).
    Returns (total, sig, inv) to allow logging both components separately.
    """
    sig = sigreg_loss(z, key)
    inv = invariance_loss(z)
    return lam * sig + (1.0 - lam) * inv, sig, inv


In [ ]:
def make_optimizer(n_steps: int, warmup_steps: int = 500):
    warmup_steps = min(warmup_steps, n_steps // 2)  # never exceed half of total steps
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=1e-3,
        warmup_steps=warmup_steps,
        decay_steps=n_steps,
        end_value=1e-5,
    )
    return optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=schedule, weight_decay=1e-4),
    )


def make_train_step(reg_type: str, optimizer, lam: float = 0.05):
    """
    Factory returning a JIT-compiled training step.

    reg_type and lam are Python constants captured in the closure so
    if/elif branches resolve at XLA trace time (no runtime branching) and
    different conditions produce independently compiled XLA programs.

    reg_type: 'lejepa' | 'vicreg' | 'pure_inv' | 'pure_sig'
    """
    @eqx.filter_jit
    def step(model, opt_state, batch, key):
        def loss_fn(model):
            z = forward(model, batch)
            if reg_type == "vicreg":
                loss = vicreg_loss(z)
                return loss, (jnp.array(0.0), loss)
            elif reg_type == "pure_inv":
                loss = invariance_loss(z)
                return loss, (jnp.array(0.0), loss)
            elif reg_type == "pure_sig":
                loss = sigreg_loss(z, key)
                return loss, (loss, jnp.array(0.0))
            else:   # lejepa
                loss, sig, inv = lejepa_loss(z, key, lam)
                return loss, (sig, inv)

        (loss, (sig, inv)), grads = eqx.filter_value_and_grad(
            loss_fn, has_aux=True
        )(model)
        updates, opt_state_new = optimizer.update(
            grads, opt_state, eqx.filter(model, eqx.is_array)
        )
        return eqx.apply_updates(model, updates), opt_state_new, loss, sig, inv

    return step


@eqx.filter_jit
def _grad_ratio_jit(model, batch, key):
    """
    Compute ||d(SIGReg)/dtheta|| / ||d(Inv)/dtheta|| without lambda scaling.
    Uses a 64-sample sub-batch to limit memory from the two extra backward passes.
    """
    def sig_fn(m): return sigreg_loss(forward(m, batch), key)
    def inv_fn(m): return invariance_loss(forward(m, batch))
    sig_g = eqx.filter_grad(sig_fn)(model)
    inv_g = eqx.filter_grad(inv_fn)(model)
    def l2_norm(g):
        leaves = jax.tree_util.tree_leaves(eqx.filter(g, eqx.is_array))
        return jnp.sqrt(sum(jnp.sum(l ** 2) for l in leaves))
    return l2_norm(sig_g) / (l2_norm(inv_g) + 1e-8)


def compute_grad_ratio(model, batch, key):
    sub = jax.tree_util.tree_map(lambda x: x[:64], batch)
    return float(_grad_ratio_jit(model, sub, key))


LOG_EVERY = 100


def run_experiment(condition: dict, n_steps: int, batch_size: int = 512):
    """
    Run one SSL training condition end-to-end.

    condition keys:
        name     : str
        reg_type : 'lejepa' | 'vicreg' | 'pure_inv' | 'pure_sig'
        lam      : float
        log_grad : bool

    All conditions share PRNGKey(0) as initial weights so differences
    in dynamics are attributable to the loss function alone.
    Returns (history dict, trained model).
    """
    name     = condition["name"]
    reg_type = condition["reg_type"]
    lam      = condition.get("lam", 0.0)
    log_grad = condition.get("log_grad", False)

    print(f"\n{'='*55}")
    print(f"  {name}   reg={reg_type}   lam={lam}")
    print(f"{'='*55}")

    model      = MiniResNet18(jax.random.PRNGKey(0))
    optimizer  = make_optimizer(n_steps)
    opt_state  = optimizer.init(eqx.filter(model, eqx.is_array))
    train_step = make_train_step(reg_type, optimizer, lam)
    loader     = make_train_loader(batch_size)
    rng        = jax.random.PRNGKey(1)

    history = dict(
        steps=[], train_loss=[], sigreg=[], inv=[], emb_var=[],
        grad_ratio_steps=[], grad_ratio=[],
    )

    for step in range(n_steps):
        batch         = next(loader)
        rng, step_key = jax.random.split(rng)
        model, opt_state, loss, sig_t, inv_t = train_step(
            model, opt_state, batch, step_key
        )

        if step % LOG_EVERY == 0:
            rng, eval_key = jax.random.split(rng)
            z = forward(model, batch)
            history["steps"].append(step)
            history["train_loss"].append(float(loss))
            history["sigreg"].append(float(sigreg_loss(z, eval_key)))
            history["inv"].append(float(invariance_loss(z)))
            history["emb_var"].append(
                float(jnp.mean(jnp.var(z.reshape(-1, z.shape[-1]), axis=0)))
            )
            if log_grad:
                rng, gr_key = jax.random.split(rng)
                history["grad_ratio_steps"].append(step)
                history["grad_ratio"].append(
                    compute_grad_ratio(model, batch, gr_key)
                )
            if step % 500 == 0:
                print(
                    f"  step {step:5d} | loss={float(loss):.4f} | "
                    f"sigreg={history['sigreg'][-1]:.4f} | "
                    f"emb_var={history['emb_var'][-1]:.6f}"
                )

    return history, model


In [ ]:
# Baseline group: 1000 steps, no linear probe.
# Plotted separately (Cell 14) to avoid y-axis compression from
# PureSIG's extreme emb_var values in the main plots.
#
# PureInv (lam=0): expected immediate collapse (emb_var -> 0).
# PureSIG (lam=1): expected emb_var >> 1 with no view alignment.
# Both use the same warmup cosine schedule as main conditions.

baseline_conditions = [
    {"name": "PureInv", "reg_type": "pure_inv", "lam": 0.0, "log_grad": False},
    {"name": "PureSIG", "reg_type": "pure_sig", "lam": 1.0, "log_grad": False},
]

baseline_results = {}
for cond in baseline_conditions:
    hist, _ = run_experiment(cond, n_steps=1000, batch_size=512)
    baseline_results[cond["name"]] = hist

print("\nBaseline group complete.")


In [ ]:
# Main experiment: 20000 steps with warmup cosine decay.
# VICReg uses original paper hyperparameters (alpha=25, beta=25, gamma=1).
# Lambda sweep covers:
#   0.01 -> collapse zone (expected emb_var -> 0)
#   0.05 -> near critical threshold
#   0.1  -> stable zone
#   0.5  -> high regularization boundary

main_conditions = [
    {"name": "VICReg",         "reg_type": "vicreg",  "lam": 0.0,  "log_grad": False},
    {"name": "LeJEPA_lam0.01", "reg_type": "lejepa",  "lam": 0.01, "log_grad": True},
    {"name": "LeJEPA_lam0.05", "reg_type": "lejepa",  "lam": 0.05, "log_grad": True},
    {"name": "LeJEPA_lam0.1",  "reg_type": "lejepa",  "lam": 0.1,  "log_grad": True},
    {"name": "LeJEPA_lam0.5",  "reg_type": "lejepa",  "lam": 0.5,  "log_grad": True},
]

main_results = {}
main_models  = {}
for cond in main_conditions:
    hist, model = run_experiment(cond, n_steps=20000, batch_size=512)
    main_results[cond["name"]] = hist
    main_models[cond["name"]]  = model

print("\nMain experiment complete.")


In [ ]:
# Save training results immediately after Cell 9 completes.
# Protects against runtime reset before linear probe or plotting is done.

with open("training_results.pkl", "wb") as f:
    pickle.dump({
        "baseline_results": baseline_results,
        "main_results":     main_results,
    }, f)
print("Training results saved to training_results.pkl")


In [ ]:
def extract_features(model, split, batch_size=512):
    """
    Extract 512-dim backbone features for all images in split.
    Uses model.encode (before proj head) — standard SSL evaluation protocol.
    Returns (feats: ndarray [N, 512], labels: ndarray [N]).
    """
    @eqx.filter_jit
    def encode_batch(imgs):
        return jax.vmap(model.encode)(imgs)

    all_feats, all_labels = [], []
    for imgs, labels in make_eval_loader(split, batch_size):
        all_feats.append(np.array(encode_batch(jnp.array(imgs))))
        all_labels.append(labels)
    return np.concatenate(all_feats), np.concatenate(all_labels)


class LinearProbe(eqx.Module):
    linear: eqx.nn.Linear

    def __init__(self, key):
        self.linear = eqx.nn.Linear(512, 10, key=key)

    def __call__(self, x):
        return self.linear(x)


def train_linear_probe(train_feats, train_labels, n_epochs=100, batch_size=1024):
    """Train a linear classifier on pre-extracted 512-dim features."""
    probe     = LinearProbe(jax.random.PRNGKey(42))
    optimizer = optax.adamw(learning_rate=1e-3, weight_decay=1e-4)
    opt_state = optimizer.init(eqx.filter(probe, eqx.is_array))

    @eqx.filter_jit
    def step(probe, opt_state, feats, labels):
        def loss_fn(probe):
            logits = jax.vmap(probe)(feats)
            return jnp.mean(
                optax.softmax_cross_entropy_with_integer_labels(logits, labels)
            )
        loss, grads = eqx.filter_value_and_grad(loss_fn)(probe)
        updates, opt_state_new = optimizer.update(
            grads, opt_state, eqx.filter(probe, eqx.is_array)
        )
        return eqx.apply_updates(probe, updates), opt_state_new, loss

    n   = len(train_feats)
    rng = np.random.default_rng(0)
    for epoch in range(n_epochs):
        idx = rng.permutation(n)
        for start in range(0, n - batch_size + 1, batch_size):
            b = idx[start:start + batch_size]
            probe, opt_state, loss = step(
                probe, opt_state,
                jnp.array(train_feats[b]),
                jnp.array(train_labels[b]),
            )
        if (epoch + 1) % 25 == 0:
            print(f"    epoch {epoch+1:3d} | loss={float(loss):.4f}")
    return probe


@eqx.filter_jit
def _eval_batch(probe, feats):
    return jnp.argmax(jax.vmap(probe)(feats), axis=-1)


def evaluate_probe(probe, feats, labels, batch_size=1024):
    correct = 0
    for start in range(0, len(feats), batch_size):
        preds   = _eval_batch(probe, jnp.array(feats[start:start+batch_size]))
        correct += int(jnp.sum(
            preds == jnp.array(labels[start:start+batch_size])
        ))
    return correct / len(labels)


def run_linear_probe(model, name, n_epochs=100):
    """Full linear probe pipeline for one trained model."""
    print(f"\n--- Linear Probe: {name} ---")
    print("  Extracting train features...")
    train_feats, train_labels = extract_features(model, "train")
    print("  Extracting test features...")
    test_feats,  test_labels  = extract_features(model, "test")
    print("  Training probe...")
    probe     = train_linear_probe(train_feats, train_labels, n_epochs=n_epochs)
    train_acc = evaluate_probe(probe, train_feats, train_labels)
    test_acc  = evaluate_probe(probe, test_feats,  test_labels)
    print(f"  Train acc: {train_acc*100:.2f}%  |  Test acc: {test_acc*100:.2f}%")
    return {"train_acc": train_acc, "test_acc": test_acc}


In [ ]:
# Run linear probe on all 5 main experiment conditions.

probe_results = {}
for name, model in main_models.items():
    probe_results[name] = run_linear_probe(model, name, n_epochs=100)

print("\nLinear probe complete.")


In [ ]:
# Baseline plot — separate figure to avoid y-axis compression in main plots.

BASELINE_COLORS = {"PureInv": "#7f7f7f", "PureSIG": "#2ca02c"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
for name, hist in baseline_results.items():
    ax.plot(hist["steps"], hist["sigreg"], label=name,
            color=BASELINE_COLORS[name], linewidth=2.0)
ax.set_xlabel("Step"); ax.set_ylabel("SIGReg Loss")
ax.set_title("Baseline Conditions: SIGReg Loss\n"
             "PureInv=no regularization  PureSIG=no invariance")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for name, hist in baseline_results.items():
    ax.plot(hist["steps"], hist["emb_var"], label=name,
            color=BASELINE_COLORS[name], linewidth=2.0)
ax.set_xlabel("Step"); ax.set_ylabel("Embedding Variance")
ax.set_title("Baseline Conditions: Embedding Variance\n"
             "collapse->0  /  over-regularization>>1")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("baseline_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to baseline_plot.png")


In [ ]:
# Main experiment plots: SIGReg loss, embedding variance, gradient ratio.

MAIN_COLORS = {
    "VICReg":          "#1f77b4",
    "LeJEPA_lam0.01":  "#ffb347",
    "LeJEPA_lam0.05":  "#d62728",
    "LeJEPA_lam0.1":   "#9467bd",
    "LeJEPA_lam0.5":   "#8c564b",
}
CORE = {"VICReg", "LeJEPA_lam0.1"}
def lw(n): return 2.5 if n in CORE else 1.5

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
for name, hist in main_results.items():
    ax.plot(hist["steps"], hist["sigreg"], label=name,
            color=MAIN_COLORS[name], linewidth=lw(name), alpha=0.9)
ax.set_xlabel("Step"); ax.set_ylabel("SIGReg Loss")
ax.set_title("Distribution Quality  (SIGReg Loss, lower is better)\n"
             "All conditions evaluated on the same scale")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
for name, hist in main_results.items():
    ax.plot(hist["steps"], hist["emb_var"], label=name,
            color=MAIN_COLORS[name], linewidth=lw(name), alpha=0.9)
ax.set_xlabel("Step"); ax.set_ylabel("Mean Embedding Variance")
ax.set_title("Embedding Variance  (collapse -> 0)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[2]
for name, hist in main_results.items():
    if not hist["grad_ratio"]:
        continue
    ax.semilogy(hist["grad_ratio_steps"], hist["grad_ratio"],
                label=name, color=MAIN_COLORS[name],
                linewidth=lw(name), alpha=0.9)
ax.set_xlabel("Step")
ax.set_ylabel("||d(SIGReg)/dtheta|| / ||d(Inv)/dtheta||")
ax.set_title("Gradient Ratio  (log scale, no lambda scaling)\n"
             "Guard dynamic: strong early, recedes as dist -> Gaussian")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to training_curves.png")


In [ ]:
# Linear probe results table.

rows = []
for name in main_results:
    hist = main_results[name]
    pr   = probe_results.get(name, {})
    rows.append({
        "Condition":          name,
        "SIGReg (final)":     f"{hist['sigreg'][-1]:.4f}",
        "Inv Loss (final)":   f"{hist['inv'][-1]:.4f}",
        "Emb Var (final)":    f"{hist['emb_var'][-1]:.4f}",
        "Grad Ratio (final)": (
            f"{hist['grad_ratio'][-1]:.4f}" if hist["grad_ratio"] else "N/A"
        ),
        "Test Acc":           f"{pr.get('test_acc', float('nan'))*100:.2f}%",
        "Train Acc":          f"{pr.get('train_acc', float('nan'))*100:.2f}%",
    })

df = pd.DataFrame(rows).set_index("Condition")
print(df.to_string())


In [ ]:
# Save complete results including probe results.
# Load this file to reproduce all tables and figures without re-running training.

with open("results.pkl", "wb") as f:
    pickle.dump({
        "baseline_results": baseline_results,
        "main_results":     main_results,
        "probe_results":    probe_results,
    }, f)
print("Final results saved to results.pkl")
